In [ ]:
import pandas as pd
import uuid
from pathlib import Path
import json

In [ ]:
RAW_DATA_DIR = Path("../data/01_raw")

In [ ]:
categorical_dtypes = {'code': str}
df_processed = pd.read_csv(RAW_DATA_DIR/"isco_clean_taxonomy.csv", dtype=categorical_dtypes)
df_processed.head()

In [ ]:
df_processed['taxonomyKey'] = 'ISCO'

In [ ]:
df_processed.rename(columns={'definition': 'text'}, inplace=True)

In [ ]:
df_processed.columns

In [ ]:

df_processed.to_csv(RAW_DATA_DIR/"labeled_training_data.csv", index=False)

In [ ]:
# Build training-style JSON payload from the taxonomy leaves

def format_code(value):
    if pd.isna(value):
        return None
    if isinstance(value, float) and value.is_integer():
        value_str = str(int(value))
    else:
        value_str = str(value)
    return value_str[:-2] if value_str.endswith('.0') else value_str


df_processed["code"] = df_processed["code"].astype(str)
df_processed["parentCode"] = df_processed["parentCode"].apply(format_code)
df_processed["level"] = df_processed["level"].astype(int)
df_processed["taxonomyKey"] = df_processed["taxonomyKey"].astype(str)

records = df_processed.to_dict(orient="records")
node_lookup = {row["code"]: row for row in records}


def build_annotations(start_row):
    """Build hierarchical annotations from leaf to root.
    
    Walks up the taxonomy tree from a leaf node to the root,
    collecting all ancestor codes along the way.
    """
    annotations = []
    current_code = start_row["code"]
    visited = set()

    while current_code and current_code not in visited:
        node = node_lookup.get(current_code)
        if not node:
            break

        level = int(node["level"])
        annotations.append({"level": level, "nodeCode": node["code"]})
        visited.add(current_code)

        # Move to parent
        parent_code = node.get("parentCode")
        if not parent_code or parent_code == "None" or pd.isna(parent_code):
            break

        current_code = parent_code

    # Sort by level (ascending) so we go from level 1 -> 2 -> 3 -> 4
    return sorted(annotations, key=lambda ann: ann["level"])


sentences = []
leaf_nodes = df_processed[df_processed["isLeaf"] == 1]
for _, row in leaf_nodes.iterrows():
    row_data = row.to_dict()
    description = "" if pd.isna(row_data["text"]) else str(row_data["text"]).strip()
    examples = "" if pd.isna(row_data["examples"]) else str(row_data["examples"]).strip()
    if examples:
        job_description = f"{description} Examples: {examples}" if description else examples
    else:
        job_description = description

    annotations = build_annotations(row_data)
    if not annotations:
        continue

    sentences.append(
        {
            "sentenceId": str(row_data["id"] or uuid.uuid4()),
            "fields": {
                "job_title": str(row_data["label"]).strip(),
                "job_description": job_description,
            },
            "annotations": annotations,
        }
    )

training_payload = {
    "taxonomyKey": df_processed["taxonomyKey"].iloc[0],
    "sentences": sentences,
}

output_path = RAW_DATA_DIR / "isco_taxonomy_sentences.json"
with open(output_path, "w") as f:
    json.dump(training_payload, f, indent=2)

print(f"✅ Generated {len(sentences)} training sentences")
print(f"📁 Saved to: {output_path}")

# Verify the output
sample = sentences[0] if sentences else None
if sample:
    print(f"\n📋 Sample annotation structure:")
    print(f"   SentenceId: {sample['sentenceId']}")
    print(f"   Title: {sample['fields']['job_title']}")
    print(f"   Levels annotated: {[ann['level'] for ann in sample['annotations']]}")
    print(f"   Codes: {[ann['nodeCode'] for ann in sample['annotations']]}")

output_path